In [ ]:
# CELL 1 - SETUP: copy the code from your private Kaggle code dataset, install it, show the GPU
import glob, importlib, os, shutil, subprocess, sys

# The code dataset is the unzipped dist/cloud_code/genai_code.zip. Kaggle may mount it under a nested folder,
# so look for the file tools/t1_overnight.py anywhere under /kaggle/input.
found = glob.glob("/kaggle/input/**/tools/t1_overnight.py", recursive=True)
assert found, "code dataset not found: add your private code dataset to this notebook (Add Input)"
CODE_DIR = os.path.dirname(os.path.dirname(found[0]))
print("code found in", CODE_DIR)

# always refresh the code (a re-run after you uploaded a new version of the code dataset must use it);
# dirs_exist_ok keeps the artifacts/ folder of an earlier run, so a resume still finds its state
shutil.copytree(CODE_DIR, "/kaggle/working/repo", dirs_exist_ok=True)
os.chdir("/kaggle/working/repo")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "-r", "requirements_kaggle.txt"])
importlib.invalidate_caches()
subprocess.call(["nvidia-smi"])


In [ ]:
# CELL 2 - PARAMETERS: the only cell you normally edit
DEVICE_PROFILE = "kaggle"     # picks configs/devices/kaggle.yaml (data under /kaggle/input, outputs in /kaggle/working)
TAG = "v2"                    # names the log/state folder artifacts/logs/<TAG>/ (keep it when you re-run to RESUME)
DRY_RUN = False               # True = tiny end-to-end check (a few minutes); set it back to False for the real run
WANDB_SECRET = "WANDB_API_KEY"   # name of the Kaggle secret that holds your Weights & Biases API key


In [ ]:
# CELL 3 - RUN: the supervised pipeline (study -> final config -> final training -> val evaluation -> ONNX -> promote)
# No logic lives in this notebook; tools/t1_overnight.py restarts crashed stages and resumes where it stopped.
import os, subprocess, sys

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret(WANDB_SECRET)
except Exception as err:
    print("No W&B API key found in the Kaggle secrets (", err, ") -> the runner will log offline.")

cmd = [sys.executable, "tools/t1_overnight.py", "--device-profile", DEVICE_PROFILE, "--tag", TAG] + (["--dry-run"] if DRY_RUN else [])
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in proc.stdout:
    print(line, end="")
print("runner exit code:", proc.wait())


In [ ]:
# CELL 4 - OUTPUTS: bundle everything you need into ONE file, then download it from the Output tab (Save Version) or the file panel
import shutil
from pathlib import Path

root, stage = Path("/kaggle/working/repo"), Path("/kaggle/working/t1_v2_results")
shutil.rmtree(stage, ignore_errors=True)
stage.mkdir(parents=True)
items = ["studies", "configs/task1_final_v2.yaml", "models", "report/tables/onnx_parity.csv",
         f"artifacts/logs/{TAG}", "artifacts/optuna", "artifacts/dryrun_overnight",    # the last one exists after a dry run
         "/kaggle/working/runs", "/kaggle/working/eval"]
for item in items:
    src = Path(item) if item.startswith("/") else root / item
    if src.exists():
        dst = stage / (src.relative_to("/kaggle/working") if item.startswith("/") else src.relative_to(root))
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst)
shutil.make_archive("/kaggle/working/t1_v2_results", "zip", stage)
print(f"{Path('/kaggle/working/t1_v2_results.zip').stat().st_size / 2**20:.1f} MB: /kaggle/working/t1_v2_results.zip")
log_dir = root / ("artifacts/dryrun_overnight/logs" if DRY_RUN else f"artifacts/logs/{TAG}")
print((log_dir / "t1_overnight_status.log").read_text()[-2500:])
